# Sea ice timeseries for ACCESS-OM3

This notebook calculates sea ice area from a single OM3 run, and compares to the NSIDC datasets for each hemisphere

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test4-d28e0359/datastore.json"

# What physical field you want, in CF terms (stays the same across models)
variable_standard_name = None # "sea_surface_salinity"

# Fallback variable name to use if the catalog doesn’t expose standard_name
fallback_variable_names = ["aice_m"]

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
# Parameters
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test4-d28e0359/datastore.json"
papermill = True
cwd = "/g/data/tm70/cyb561/repos/access-om3-paper-test2/notebooks/mkfigs_output_MC_25km_jra_iaf+wombatlite-test4-d28e0359/"
nbname = "SeaIce_area.ipynb"


In [ ]:
import os
if not papermill: 
    import nci_ipynb  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)
plotfolder = str(cwd) + "/" if cwd else "./"

from exptdata_access import get_experiment_info, guess_experiment_from_esm_file
from model_agnostic import get_lon_lat_from_catalog, select_variable
# Infer model information from the datastore path
expt_key, info = guess_experiment_from_esm_file(esm_file)
model_name = info["model"]             # OM3 or CM3

In [ ]:
import intake
from xarray import DataTree, map_over_datasets

from dask.distributed import Client

import xarray as xr
import numpy as np
from datetime import timedelta
import cf_xarray as cfxr
import xesmf

# plotting
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cmocean.cm as cmo
import matplotlib.lines as mlines

In [ ]:
IAF = esm_file.find('iaf') > 0

In [ ]:
client = Client(threads_per_worker=1)
print(client.dashboard_link)

In [ ]:
if IAF:
    t_slice = slice('2009','2018') #manually pick some years in both om2 & 3 experiments
else:
    years = np.unique(ds.time.dt.year)[-10:]
    t_slice = slice(str(years[0]),str(years[-1]))
    t_slice

# Data

## Open the intake-esm datastore & model dataset for sea ice concentration

In [ ]:
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)


In [ ]:
def available_variables(datastore):
    """Return a pandas dataframe summarising the variables in a datastore"""
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

In [ ]:
variable = "aice_m"
area_variable = "tarea"

In [ ]:
dset = datastore.search(variable=variable, frequency="1mon").to_dask()
dset = dset.chunk({'time':12})
tarea = datastore.search(variable=area_variable, frequency="fx", realm="seaIce", path = ".*output000.*").to_dask()

## Observations

In [ ]:
sh_obs = xr.open_mfdataset('/g/data/av17/access-nri/OM3/CDR_G02202_V6/south/monthly/sic_pss25*.nc', data_vars='minimal', compat='override', coords='minimal', parallel=True).chunk({'time':12})

In [ ]:
nh_obs = xr.open_mfdataset('/g/data/av17/access-nri/OM3/CDR_G02202_V6/north/monthly/sic_psn25*.nc', data_vars='minimal', compat='override', coords='minimal', parallel=True).chunk({'time':12})

In [ ]:
def add_nsidc_areas(filename, obs_ds):
    """
    Load the unformatted binary file with grid areas from NSIDC, and add as a coordinate to an xarray dataset
    """
    areasNd = np.fromfile(filename, dtype=np.int32).reshape(
        obs_ds.cdr_seaice_conc_monthly.isel(time=0).shape
    )
    areasKmNd_sh = areasNd /1000 #convert to km^2

    obs_ds["area"] = xr.DataArray(areasKmNd_sh, dims=["y", "x"])

    obs_ds.set_coords("area")
        
    return obs_ds

In [ ]:
sh_obs = add_nsidc_areas("/g/data/av17/access-nri/OM3/CDR_G02202_V5/NSIDC-G02202-sh/pss25area_v3.dat", sh_obs)
nh_obs = add_nsidc_areas("/g/data/av17/access-nri/OM3/CDR_G02202_V5/NSIDC-G02202-nh/psn25area_v3.dat", nh_obs)

# Calculation

In [ ]:
sic = dset[variable]
area = tarea[area_variable].load()

antarctic_seaice = (sic * area).sel(nj=slice(0,int(len(dset.nj)/2))).sum(['nj','ni'])
arctic_seaice = (sic * area).sel(nj=slice(int(len(dset.nj)/2), None)).sum(['nj','ni'])

sh_area = antarctic_seaice.compute()
nh_area = arctic_seaice.compute()

In [ ]:
sh_sic = sh_obs['cdr_seaice_conc_monthly']
sharea = sh_obs.area.load()

nh_sic = nh_obs['cdr_seaice_conc_monthly']
nharea = nh_obs.area.load()

antarctic_seaice_obs = (sh_sic * sharea).sum(['y','x'])
arctic_seaice_obs = (nh_sic * nharea).sum(['y','x'])

sh_area_obs = antarctic_seaice_obs.compute()
nh_area_obs = arctic_seaice_obs.compute()

In [ ]:
nh_area_obs.loc[{"time": "1984-07-01"}] = np.nan
nh_area_obs.loc[{"time": "1988-01-01"}] = np.nan
nh_area_obs.loc[{"time": "1987-12"}] = np.nan

sh_area_obs.loc[{"time": "1988-01-01"}] = np.nan
sh_area_obs.loc[{"time": "1987-12"}] = np.nan

# Plot

In [ ]:
font = {'size':13}
tick_font=12

In [ ]:
plt.figure(1,(12,5))

plt.subplot(1,2,1)
(nh_area.sel(time=t_slice).groupby('time.month').mean('time') / 1e12).plot(label = 'OM3',c='b')      # convert from m^2 -> 10^6 km^2
(nh_area_obs.sel(time=t_slice).groupby('time.month').mean('time') / 1e6).plot(label = 'Observations',c='k')      # convert from km^2 -> 10^6 km^2

plt.xlabel('Month',font)
plt.ylabel('Sea Ice Area (10$^6$ km$^2$)',font)
plt.title('Arctic',font)

plt.subplot(1,2,2)
(sh_area.sel(time=t_slice).groupby('time.month').mean('time') / 1e12).plot(label = 'OM3',c='b')   # convert from m^2 -> 10^6 km^2
(sh_area_obs.sel(time=t_slice).groupby('time.month').mean('time') / 1e6).plot(label = 'Observations',c='k')   # convert from km^2 -> 10^6 km^2

plt.xlabel('Month',font)
plt.ylabel('Sea Ice Area (10$^6$ km$^2$)',font)
plt.title('Antarctic',font)

plt.legend(prop=font,loc='upper left', bbox_to_anchor=(1, 1))


figfile = plotfolder + f'seaice_area_climatology.jpg'
plt.savefig(figfile, dpi=300, bbox_inches='tight') 

plt.show()
mkmd.savefig(plt.gcf(), "Sea Ice Area", "Seasonal sea ice area for Arctic and Antarctic. [GitHub issue: Sea ice annual cycle](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/21)", dpi=300)

In [ ]:
plt.figure(1,(12,5))

plt.subplot(1,2,1)
(nh_area.groupby('time.year').mean('time') / 1e12).plot(label = 'OM3',c='b')      # convert from m^2 -> 10^6 km^2
(nh_area_obs.groupby('time.year').mean('time')[:-1] / 1e6).plot(label = 'Observations',c='k')      # convert from km^2 -> 10^6 km^2

plt.xlabel('Year',font)
plt.ylabel('Sea Ice Area (10$^6$ km$^2$)',font)
plt.title('Arctic',font)

plt.subplot(1,2,2)
(sh_area.groupby('time.year').mean('time') / 1e12).plot(label = 'OM3',c='b')   # convert from m^2 -> 10^6 km^2
(sh_area_obs.groupby('time.year').mean('time')[:-1] / 1e6).plot(label = 'Observations',c='k')   # convert from km^2 -> 10^6 km^2

plt.xlabel('Year',font)
plt.ylabel('Sea Ice Area (10$^6$ km$^2$)',font)
plt.title('Antarctic',font)

plt.legend(prop=font,loc='upper left', bbox_to_anchor=(1, 1))

figfile = plotfolder + f'seaice_area_annual_mean.jpg'
plt.savefig(figfile, dpi=300, bbox_inches='tight') 

plt.show()
mkmd.savefig(plt.gcf(), "Sea Ice Area", "Annual mean sea ice area timeseries for Arctic and Antarctic. [GitHub issue: Sea ice extent time series](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/20)", dpi=300)

In [ ]:
# 12-mo running mean minimum, mean and maximum of area for all models
plt.figure(1,(12,5))

plt.subplot(1,2,1)
v = nh_area/1e12
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).max()[6:-5], color='b', linewidth=1, label='OM3')
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).mean()[6:-5], color='b', linewidth=1)
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).min()[6:-5], color='b', linewidth=1)

plt.subplot(1,2,2)
v = sh_area/1e12
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).max()[6:-5], color='b', linewidth=1, label='OM3')
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).mean()[6:-5], color='b', linewidth=1)
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).min()[6:-5], color='b', linewidth=1)

plt.subplot(1,2,1)
v = nh_area_obs/1e6
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).max()[6:-5], color='k', linewidth=1, label='Observations')
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).mean()[6:-5], color='k', linewidth=1)
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).min()[6:-5], color='k', linewidth=1)

plt.subplot(1,2,2)
v = sh_area_obs/1e6
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).max()[6:-5], color='k', linewidth=1, label='Observations')
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).mean()[6:-5], color='k', linewidth=1)
plt.plot(v['time'][6:-5],v.rolling(time=12, center=True).min()[6:-5], color='k', linewidth=1)

plt.subplot(1,2,1)
plt.ylim(ymin=-0.5,ymax=18)
plt.grid(axis='y')
plt.xlabel('Year',font)
plt.ylabel(r'Sea ice area (10$^{12}$ m$^2$)',font)
plt.title('Arctic ice area minimum, mean and maximum',font)
plt.subplot(1,2,2)
plt.ylim(ymin=-0.5,ymax=18)
plt.grid(axis='y')
plt.xlabel('Year',font)
plt.ylabel(r'Sea ice area (10$^{12}$ m$^2$)',font)
plt.title('Antarctic ice area minimum, mean and maximum',font)
plt.legend(prop=font,loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()

figfile = plotfolder + f'seaice_area_rolling_min_mean_max.jpg'
plt.savefig(figfile, dpi=300, bbox_inches='tight') 

plt.show()
mkmd.savefig(plt.gcf(), "Sea Ice Area", "Sea ice area minimum, mean and maximum timeseries. [GitHub issue: Sea ice extent time series](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/20)", dpi=300)

In [ ]:
client.close()